# TinyBanglaBERT – full re-run on Kaggle
**Before running:** Settings → Accelerator = **GPU T4 x2 / P100**, Internet = **On**.
Add the dataset made from `tiny_bert-fixed.zip` via *Add Input*. Then use *Save Version → Save & Run All*.

In [ ]:
import glob, os, shutil, subprocess, sys
# 1) locate the uploaded project and copy it to the writable disk
hits = glob.glob('/kaggle/input/**/tiny_bert-main', recursive=True)
assert hits, "Upload tiny_bert-fixed.zip as a Kaggle dataset and add it to this notebook"
shutil.rmtree('/kaggle/working/tiny_bert-main', ignore_errors=True)
shutil.copytree(hits[0], '/kaggle/working/tiny_bert-main')
os.chdir('/kaggle/working/tiny_bert-main')
print(os.getcwd())

In [ ]:
!pip install -q omegaconf lime shap onnx onnxruntime onnxscript optuna statsmodels tabulate
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Data splits (fixed seed 42)
Expect 12,044 train / 1,506 val / 1,506 test.

In [ ]:
!python -m scripts.prepare_data --config configs/base.yaml --seed 42
import json; m = json.load(open('data/processed/splits.json'))['metadata']; print({k:m[k] for k in ('train_size','val_size','test_size','total_unlabeled')})

## 2. Teacher (BanglaBERT) — recreates the missing weights + soft labels

In [ ]:
!python -m scripts.train --config configs/teacher_baseline.yaml --seed 42
print(sorted(os.listdir('experiments/teacher')))
assert os.path.exists('experiments/teacher/train_soft_labels.pt'), "teacher soft labels missing"
print(open('experiments/teacher/metrics.json').read()[:600])

## 3. KD hyper-parameter search (now genuinely distilling)
Uses the pretrained small-subset init. The grid should no longer be flat.

In [ ]:
!python -m scripts.tune_kd --config configs/student_kd_improved.yaml --init small_subset --epochs 3
best = json.load(open('experiments/tuning/kd_grid_results.json'))['best_params']; print(best)
vals = {round(r['val_f1_macro'],4) for r in json.load(open('experiments/tuning/kd_grid_results.json'))['grid_results']}
print('distinct val F1 values in grid:', len(vals), '(if this is 1 or 2, KD is still not active – stop and report)')

## 4. Improved student (pretrained init + KD + unlabeled pool)

In [ ]:
T, A = best['temperature'], best['alpha']
!python -m scripts.train --config configs/student_kd_improved.yaml --seed 42 --temperature {T} --alpha {A}
print(open('experiments/student_kd_improved/test_metrics.json').read())

## 5. Re-run the 2×2 ablation (old 'distilled' rows were really supervised)

In [ ]:
for c in ['student_random_baseline','student_random_distilled','student_teacher_init_baseline','student_teacher_init_distilled']:
    !python -m scripts.train --config configs/{c}.yaml --seed 42

## 6. Summary of all test-set results

In [ ]:
import pandas as pd
rows=[]
for run in ['teacher','student_random_baseline','student_random_distilled','student_teacher_init_baseline','student_teacher_init_distilled','student_kd_improved']:
    for f in ('test_metrics.json','metrics.json'):
        p=f'experiments/{run}/{f}'
        if os.path.exists(p):
            d=json.load(open(p)); rows.append({'run':run,'accuracy':d.get('accuracy'),'f1_macro':d.get('f1_macro'),'f1_clickbait':d.get('f1_clickbait')}); break
df=pd.DataFrame(rows); print(df.round(4).to_string(index=False)); df.to_csv('experiments/rerun_summary.csv',index=False)

In [ ]:
# 7) package everything you must keep (Kaggle only persists /kaggle/working)
!cd /kaggle/working/tiny_bert-main && zip -qr /kaggle/working/experiments_rerun.zip experiments data/processed -x "*/checkpoints/*"
!ls -lh /kaggle/working/experiments_rerun.zip